# 再現の確認（shibayama_novelty）

`src/shibayama_novelty/README.md` の工程 1〜13 を順に流し、`data/derived/shibayama_novelty/` の全ファイルがコミットされたものとバイト単位で一致するかを確かめる。

- どこで開いてもよい。リポジトリの一番上を探し、スクリプトはそこから流す。
- 工程2と5は scispaCy の語ベクトル表を `data/cache/scispacy/` から読む。無ければ工程2が配布元から落とす（初回だけ、約530MB・数分）。工程2の出力が一致すれば、落としたものが正しいモデルだと確かめられたことになる。
- 流す前に作業ツリーの `data/derived/shibayama_novelty/` がコミットされた状態であることを前提にする（流す前のハッシュを基準にする）。

In [ ]:
import hashlib
import subprocess
import sys
from pathlib import Path

# notebooks/reproduce/ で開いても動くよう、上にたどってリポジトリの一番上を探す
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "shibayama_novelty" / "compute_shibayama_novelty.py").exists())
DERIVED = ROOT / "data" / "derived" / "shibayama_novelty"

STEPS = [
    "clean_reference_titles",
    "embed_titles_scispacy",
    "compute_shibayama_novelty",
    "build_title_distance_pairs",
    "validate_word_vectors",
    "validate_distance_shared_refs",
    "validate_distance_field_labels",
    "plot_novelty_distributions",
    "validate_novelty_citations",
    "plot_novelty_citations",
    "regress_citations_on_novelty",
    "plot_citation_distribution",
    "compare_novelty_sciscinet",
]


def snapshot() -> dict[str, str]:
    return {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(DERIVED.rglob("*")) if p.is_file() and p.name != ".DS_Store"}


before = snapshot()
print(f"流す前: {len(before)} ファイル")

In [ ]:
for i, name in enumerate(STEPS, start=1):
    print(f"[{i}] {name}")
    res = subprocess.run([sys.executable, f"src/shibayama_novelty/{name}.py"], cwd=ROOT,
                         capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"工程{i} {name} が失敗した")

In [ ]:
after = snapshot()
changed = sorted(k for k in before.keys() & after.keys() if before[k] != after[k])
added = sorted(after.keys() - before.keys())
removed = sorted(before.keys() - after.keys())
for label, items in [("変わった", changed), ("増えた", added), ("消えた", removed)]:
    for k in items:
        print(f"{label}: {k}")
print(f"一致 {len(before.keys() & after.keys()) - len(changed)} / {len(after)} ファイル")
assert not (changed or added or removed), "コミットされた出力と一致しない"
print("✅ すべての出力がコミットされたものと一致した")